# 🛰️ Monitoramento de Queimadas em UCs do Pará (NGEO / IDEFLOR-Bio)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NGEO-IDEFLOR-Bio/cartografia-queimadas-ucs-pa/blob/main/notebooks/colab_runner.ipynb)

Painel automatizado para detecção de focos de calor (**BDQueimadas / INPE**) e análise de cicatrizes de fogo com **Sentinel-2 Harmonized (Google Earth Engine)** nas 29 Unidades de Conservação Estaduais do Pará.

Gera automaticamente 3 pranchas no padrão oficial **A4 Paisagem (297 x 210 mm)**:
1. **Falsa Cor (SWIR-NIR-Red)**: Realce de cicatrizes recentes e frentes de queima ativa.
2. **Cor Natural (RGB)**: Mosaico orbital sem nuvens.
3. **NBR (Normalized Burn Ratio)**: Índice espectral de severidade do fogo.

In [ ]:
#@title 1. Instalação de Dependências e Setup do Ambiente { display-mode: "form" }
#@markdown *Clique no botão 'Play' para preparar o ambiente e baixar o repositório.*

!apt-get update -qq && apt-get install -y -qq libreoffice-writer > /dev/null 2>&1
!pip install -q earthengine-api geopandas rasterio matplotlib pillow python-docx openpyxl

import os
import sys
from pathlib import Path

REPO_NAME = "cartografia-queimadas-ucs-pa"
if not Path(REPO_NAME).exists() and not Path("src").exists():
    print(f"Clonando o repositório oficial {REPO_NAME}...")
    !git clone https://github.com/NGEO-IDEFLOR-Bio/cartografia-queimadas-ucs-pa.git

if Path(REPO_NAME).exists():
    os.chdir(REPO_NAME)

sys.path.append(os.getcwd())
print(f"[OK] Ambiente preparado e repositório configurado: {os.getcwd()}")


In [ ]:
#@title 2. Autenticação no Google Earth Engine (GEE) { display-mode: "form" }
GCP_PROJECT = "ee-samuelsantosambientalcourse" #@param {type:"string"}
#@markdown > ⚠️ **Atenção colegas:** Altere o campo acima para o ID do seu próprio projeto Google Cloud habilitado no Earth Engine.

import ee

try:
    ee.Initialize(project=GCP_PROJECT)
    print(f"[OK] Google Earth Engine inicializado com sucesso no projeto: {GCP_PROJECT}")
except Exception as e:
    print(f"! Solicitando autorização de acesso ao GEE...")
    ee.Authenticate()
    ee.Initialize(project=GCP_PROJECT)
    print(f"[OK] Google Earth Engine autenticado e inicializado com sucesso no projeto: {GCP_PROJECT}!")

In [ ]:
#@title 3. Parâmetros de Análise Cartográfica { display-mode: "form" }
NOME_UC = "Área de Proteção Ambiental do Marajó" #@param ["Área de Proteção Ambiental do Marajó", "Área de Proteção Ambiental da Ilha de Combu", "Área de Proteção Ambiental do Lago de Tucuruí", "Área de Proteção Ambiental Triunfo do Xingu", "Área de Proteção Ambiental de Algodoal-Maiandeua", "Área de Proteção Ambiental de São Geraldo do Araguaia – APA Araguaia", "Área de Proteção Ambiental da Região Metropolitana de Belém", "Área de Proteção Ambiental Paytuna", "Parque Estadual Monte Alegre – PEMA", "Parque Estadual Charapucu", "Parque Estadual da Serra dos Martírios/Andorinhas", "Parque Estadual do Utinga “Camillo Vianna”", "Parque Estadual Ambiental das Árvores Gigantes da Amazônia (PAGAM)", "Floresta Estadual do Paru", "Floresta Estadual do Trombetas", "Floresta Estadual do Iriri", "Flota Estadual de Faro", "Estação Ecológica do Grão-Pará (ESEC)", "Estação Ecológica Mamuru (ESEC)", "Reserva Biológica do Maicuru (REBIO)", "Refúgio de Vida Silvestre Metrópole da Amazônia (REVIS)", "Refúgio de Vida Silvestre Padre Sérgio Tonetto", "Refúgio de Vida Silvestre Rios São Benedito e Azul  (REVIS)", "Refúgio de Vida Silvestre Tabuleiro do Embaubal (REVIS)", "Reserva de Desenvolvimento Sustentável Alcobaça", "Reserva de Desenvolvimento Sustentável Campo das Mangabas", "Reserva de Desenvolvimento Sustentável Pucuruí-Ararão", "Reserva de Desenvolvimento Sustentável Vitória de Souzel", "Monumento Natural Atalaia (MONA)"] {allow-input: true}
DIAS_MONITORAMENTO = 30 #@param {type:"slider", min:7, max:30, step:1}
#@markdown *O monitoramento analisa automaticamente a janela recente (ex: últimos 30 dias até hoje) direto do INPE/BDQueimadas sem necessidade de baixar arquivos.*

from datetime import date, timedelta
hoje = date.today()
data_inicio_calc = (hoje - timedelta(days=DIAS_MONITORAMENTO)).strftime("%d/%m/%Y")
data_fim_calc = hoje.strftime("%d/%m/%Y")

print(f"Configuração ativa:")
print(f" • Unidade de Conservação: {NOME_UC}")
print(f" • Janela de Monitoramento: Últimos {DIAS_MONITORAMENTO} dias ({data_inicio_calc} até {data_fim_calc})")
print(f" • Fonte dos Focos: BDQueimadas/INPE (Download 100% Automático)")

In [ ]:
#@title 4. Executar Pipeline e Gerar as Pranchas Oficiais { display-mode: "form" }
#@markdown *Clique no 'Play' para iniciar o processamento integrado (Sentinel-2 + BDQueimadas + Layout A4 + Relatório Word + Excel + KML).*

from src.exporter import processar_e_exportar_todos_os_mapas

resultado = processar_e_exportar_todos_os_mapas(
    nome_uc=NOME_UC,
    dias_analise=DIAS_MONITORAMENTO,
    focos_source="BDQUEIMADAS"
)


In [ ]:
#@title 5. Visualizar Pranchas Cartográficas e Links de Arquivos { display-mode: "form" }

if 'resultado' not in globals():
    raise RuntimeError("⚠️ Atenção: Por favor, execute primeiro a Célula '4. Executar Pipeline' antes de visualizar os mapas.")

from pathlib import Path
from IPython.display import Image, HTML, display

pranchas = resultado.get("pranchas", {})
uc_nome = resultado.get("uc", "Unidade de Conservação")

print("\n" + "="*70)
print(f"🗺️ PRANCHAS CARTOGRÁFICAS GERADAS – {uc_nome.upper()}")
print("="*70)

for chave, info in pranchas.items():
    titulo_mapa = info["titulo"].upper()
    png_path = info.get("png", "")
    pdf_path = info.get("pdf", "")
    print(f"\n--- {titulo_mapa} ---")
    print(f"  • PNG: {png_path}")
    print(f"  • PDF: {pdf_path}")
    if png_path and Path(png_path).exists():
        display(Image(filename=png_path, width=950))

# Exibir links clicáveis em HTML leve
rel_docx = resultado.get("relatorio_docx", "")
rel_pdf = resultado.get("relatorio_pdf", "")
planilha_xlsx = resultado.get("planilha_excel", "")
kml_path = resultado.get("pacote_kml", "")

itens_html = []
if rel_docx and Path(rel_docx).exists():
    itens_html.append(f'<li>📄 <b>Relatório Word (.docx):</b> <a href="{rel_docx}" download="{Path(rel_docx).name}" target="_blank">{Path(rel_docx).name}</a></li>')
if rel_pdf and Path(rel_pdf).exists():
    itens_html.append(f'<li>📄 <b>Relatório PDF (.pdf):</b> <a href="{rel_pdf}" download="{Path(rel_pdf).name}" target="_blank">{Path(rel_pdf).name}</a></li>')
if planilha_xlsx and Path(planilha_xlsx).exists():
    itens_html.append(f'<li>📊 <b>Planilha Excel (.xlsx):</b> <a href="{planilha_xlsx}" download="{Path(planilha_xlsx).name}" target="_blank">{Path(planilha_xlsx).name}</a></li>')
if kml_path and Path(kml_path).exists():
    itens_html.append(f'<li>🌍 <b>Pacote Google Earth (.kml):</b> <a href="{kml_path}" download="{Path(kml_path).name}" target="_blank">{Path(kml_path).name}</a></li>')

for p_info in pranchas.values():
    pdf_p = p_info.get("pdf")
    if pdf_p and Path(pdf_p).exists():
        itens_html.append(f'<li>🗺️ <b>Prancha PDF:</b> <a href="{pdf_p}" download="{Path(pdf_p).name}" target="_blank">{Path(pdf_p).name}</a></li>')

html_box = f'''
<div style="background-color: #F8F9F9; border: 1.5px solid #2980B9; border-radius: 6px; padding: 14px 20px; margin: 20px 0; font-family: Arial, sans-serif;">
  <h4 style="margin: 0 0 10px 0; color: #154360; font-size: 15px;">📥 LINKS PARA DOWNLOAD INDIVIDUAL DOS ARQUIVOS:</h4>
  <ul style="line-height: 1.8; font-size: 13px; color: #2C3E50; margin: 0; padding-left: 20px;">
    {''.join(itens_html)}
  </ul>
  <p style="margin: 12px 0 0 0; font-size: 12px; color: #566573; font-style: italic;">
    💡 <b>Dica:</b> Para baixar todos os arquivos de uma única vez em arquivo compactado, execute a <b>Célula 6</b> abaixo.
  </p>
</div>
'''
display(HTML(html_box))


In [ ]:
#@title 6. Baixar Todos os Arquivos em Arquivo Compactado (.ZIP) { display-mode: "form" }
#@markdown *Execute esta célula para gerar um arquivo .ZIP contendo todas as pranchas (PNG e PDF), relatório, Excel e KML, e iniciar o download direto no seu computador.*

import zipfile
from pathlib import Path

if 'resultado' not in globals():
    raise RuntimeError("⚠️ Execute primeiro a Célula 4 antes de tentar baixar os arquivos.")

uc_nome = resultado.get("uc", "UC")
safe_name = uc_nome.replace(" ", "_").replace("/", "-")
zip_filename = f"Pacote_Monitoramento_{safe_name}.zip"
zip_path = Path("outputs") / zip_filename

# Coleta todos os arquivos gerados
arquivos_para_zip = []

# Pranchas Cartográficas (PNG e PDF)
for p_info in resultado.get("pranchas", {}).values():
    if p_info.get("png") and Path(p_info["png"]).exists():
        arquivos_para_zip.append(Path(p_info["png"]))
    if p_info.get("pdf") and Path(p_info["pdf"]).exists():
        arquivos_para_zip.append(Path(p_info["pdf"]))

# Relatório, Planilha e KML
for chave in ["relatorio_docx", "relatorio_pdf", "planilha_excel", "pacote_kml"]:
    f_str = resultado.get(chave)
    if f_str and Path(f_str).exists():
        arquivos_para_zip.append(Path(f_str))

# Compacta em arquivo ZIP
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for f in arquivos_para_zip:
        zipf.write(f, arcname=f.name)

tamanho_mb = zip_path.stat().st_size / (1024 * 1024)
print(f"✅ Pacote ZIP gerado com sucesso: {zip_path.name} ({tamanho_mb:.2f} MB)")
print(f"📦 Total de {len(arquivos_para_zip)} arquivos incluídos no pacote:")
for f in arquivos_para_zip:
    print(f"   • {f.name}")

# Dispara download no Google Colab
try:
    from google.colab import files
    print("\nIniciando download do arquivo ZIP no seu navegador...")
    files.download(str(zip_path))
except Exception:
    print(f"\n💡 Ambiente local: O arquivo ZIP está pronto em: {zip_path.resolve()}")
